# ============================================
# 12_combined_cycle_power_plant_downloader.ipynb
# UCI Combined Cycle Power Plant Dataset
# ============================================

## Dataset Title
UCI Combined Cycle Power Plant Dataset

## Source URL
Official dataset page:  
https://archive.ics.uci.edu/dataset/294/combined+cycle+power+plant

## Purpose of This Dataset
This dataset is used as a real power-plant regression benchmark dataset in the PhD workflow.

It is especially useful for:
- energy-output regression
- baseline benchmarking
- model comparison
- reproducibility experiments
- final pipeline validation

## Why We Need This Dataset
We need this dataset because it provides a clean and well-known real-world energy prediction benchmark.

According to the official UCI page:
- it contains 9,568 data points
- it covers 6 years (2006–2011)
- it has 4 input features
- it predicts net hourly electrical energy output
- the downloadable files are provided in `.xlsx` and `.ods` formats
- the dataset has no missing values

This makes it highly suitable for:
- Notebook 2: Baseline Models
- Notebook 3: NEO-Forecast
- Notebook 11: Full System Simulation support
- Notebook 12: Reproducibility and benchmarking

## What We Are Doing in This Notebook
In this notebook, we will:

1. Mount Google Drive
2. Create dataset folders
3. Download the dataset using the official UCI Python-access method
4. Save the features and targets as CSV files
5. Save metadata and variable information
6. Save a dataset manifest
7. Create a clean ZIP bundle
8. Save everything in Google Drive
9. Create a final package ZIP
10. Download that final package to the local PC

## Where This Data Will Be Used Next
This dataset will be used in:

- Notebook 2: Baseline Models
- Notebook 3: NEO-Forecast
- Notebook 11: Full System Simulation support
- Notebook 12: Reproducibility and benchmarking

## Notes
The official UCI page states that this dataset contains 9,568 data points collected from a combined cycle power plant over 6 years, with four ambient variables used to predict the net hourly electrical energy output, and that the data are available in `.xlsx` and `.ods` forms. :contentReference[oaicite:1]{index=1}

In [1]:
# ============================================
# 12_combined_cycle_power_plant_downloader.ipynb
# FULL FIXED SINGLE CELL
# ============================================

import json
import shutil
import zipfile
from pathlib import Path

import pandas as pd
from google.colab import drive, files

# -----------------------------
# 1. Dataset metadata
# -----------------------------
DATASET_NAME = "UCI Combined Cycle Power Plant Dataset"
SOURCE_PAGE = "https://archive.ics.uci.edu/dataset/294/combined+cycle+power+plant"
UCI_ID = 294

print("Dataset Name :", DATASET_NAME)
print("Source Page  :", SOURCE_PAGE)
print("UCI ID       :", UCI_ID)

# -----------------------------
# 2. Mount Google Drive
# -----------------------------
drive.mount('/content/drive')

# -----------------------------
# 3. Create folder structure
# -----------------------------
project_root = Path("/content/drive/MyDrive/PhD_Thesis_Datasets")
dataset_root = project_root / "12_combined_cycle_power_plant"
raw_dir = dataset_root / "raw"
processed_dir = dataset_root / "processed"
bundle_dir = dataset_root / "bundle"

for folder in [project_root, dataset_root, raw_dir, processed_dir, bundle_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nCreated folders:")
print(" -", project_root)
print(" -", dataset_root)
print(" -", raw_dir)
print(" -", processed_dir)
print(" -", bundle_dir)

# -----------------------------
# 4. Install UCI helper package
# -----------------------------
print("\nInstalling ucimlrepo if needed...")
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ucimlrepo"])

from ucimlrepo import fetch_ucirepo

# -----------------------------
# 5. Fetch dataset from official UCI API path
# -----------------------------
print("\nFetching dataset from UCI...")
dataset = fetch_ucirepo(id=UCI_ID)

X = dataset.data.features
y = dataset.data.targets
metadata = dataset.metadata
variables = dataset.variables

print("Features shape:", X.shape)
print("Targets shape :", y.shape)

# -----------------------------
# 6. Save raw and processed CSV files
# -----------------------------
features_csv = processed_dir / "ccpp_features.csv"
targets_csv = processed_dir / "ccpp_targets.csv"
full_csv = processed_dir / "ccpp_full_dataset.csv"
metadata_json = raw_dir / "ccpp_metadata.json"
variables_csv = raw_dir / "ccpp_variables.csv"

X.to_csv(features_csv, index=False)
y.to_csv(targets_csv, index=False)
pd.concat([X, y], axis=1).to_csv(full_csv, index=False)

with open(metadata_json, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, default=str)

variables.to_csv(variables_csv, index=False)

print("\nSaved processed files:")
print(" -", features_csv)
print(" -", targets_csv)
print(" -", full_csv)
print(" -", metadata_json)
print(" -", variables_csv)

# -----------------------------
# 7. Save manifest
# -----------------------------
manifest = {
    "dataset_name": DATASET_NAME,
    "source_page": SOURCE_PAGE,
    "uci_id": UCI_ID,
    "saved_files": [
        str(features_csv),
        str(targets_csv),
        str(full_csv),
        str(metadata_json),
        str(variables_csv),
    ],
    "used_in_next_notebooks": [
        "Notebook 2: Baseline Models",
        "Notebook 3: NEO-Forecast",
        "Notebook 11: Full System Simulation support",
        "Notebook 12: Reproducibility and benchmarking"
    ]
}

manifest_path = dataset_root / "dataset_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved manifest:", manifest_path)

# -----------------------------
# 8. Create clean bundle ZIP
# -----------------------------
clean_bundle_zip_path = bundle_dir / "combined_cycle_power_plant_CLEAN_bundle.zip"

bundle_files = [features_csv, targets_csv, full_csv, metadata_json, variables_csv, manifest_path]

with zipfile.ZipFile(clean_bundle_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in bundle_files:
        if fp == metadata_json or fp == variables_csv:
            arcname = "raw/" + fp.name
        elif fp == manifest_path:
            arcname = "manifest/" + fp.name
        else:
            arcname = "processed/" + fp.name
        zf.write(fp, arcname=arcname)

print("Created clean bundle ZIP:", clean_bundle_zip_path)
print("Clean bundle size:", round(clean_bundle_zip_path.stat().st_size / 1024, 2), "KB")

# -----------------------------
# 9. Create full package ZIP in Google Drive
# -----------------------------
full_zip_path = dataset_root / "combined_cycle_power_plant_FULL_PACKAGE.zip"

with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in [features_csv, targets_csv, full_csv]:
        zf.write(fp, arcname="processed/" + fp.name)
    zf.write(metadata_json, arcname="raw/" + metadata_json.name)
    zf.write(variables_csv, arcname="raw/" + variables_csv.name)
    zf.write(clean_bundle_zip_path, arcname="bundle/" + clean_bundle_zip_path.name)
    zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Created full package ZIP in Google Drive:", full_zip_path)
print("Full package size:", round(full_zip_path.stat().st_size / 1024, 2), "KB")

# -----------------------------
# 10. Copy final ZIP to /content for reliable PC download
# -----------------------------
runtime_copy = Path("/content/combined_cycle_power_plant_FULL_PACKAGE.zip")
shutil.copy2(full_zip_path, runtime_copy)

print("Runtime backup created:", runtime_copy)
print("Runtime backup size:", round(runtime_copy.stat().st_size / 1024, 2), "KB")

# -----------------------------
# 11. Final summary
# -----------------------------
print("\n===== SUMMARY =====")
print("Dataset root         :", dataset_root)
print("Processed folder     :", processed_dir)
print("Manifest             :", manifest_path)
print("Clean bundle ZIP     :", clean_bundle_zip_path)
print("Full package ZIP     :", full_zip_path)
print("Runtime backup ZIP   :", runtime_copy)
print("===================")

# -----------------------------
# 12. Download final package to PC
# -----------------------------
print("\nStarting browser download...")
files.download(str(runtime_copy))

Dataset Name : UCI Combined Cycle Power Plant Dataset
Source Page  : https://archive.ics.uci.edu/dataset/294/combined+cycle+power+plant
UCI ID       : 294
Mounted at /content/drive

Created folders:
 - /content/drive/MyDrive/PhD_Thesis_Datasets
 - /content/drive/MyDrive/PhD_Thesis_Datasets/12_combined_cycle_power_plant
 - /content/drive/MyDrive/PhD_Thesis_Datasets/12_combined_cycle_power_plant/raw
 - /content/drive/MyDrive/PhD_Thesis_Datasets/12_combined_cycle_power_plant/processed
 - /content/drive/MyDrive/PhD_Thesis_Datasets/12_combined_cycle_power_plant/bundle

Installing ucimlrepo if needed...

Fetching dataset from UCI...
Features shape: (9568, 4)
Targets shape : (9568, 1)

Saved processed files:
 - /content/drive/MyDrive/PhD_Thesis_Datasets/12_combined_cycle_power_plant/processed/ccpp_features.csv
 - /content/drive/MyDrive/PhD_Thesis_Datasets/12_combined_cycle_power_plant/processed/ccpp_targets.csv
 - /content/drive/MyDrive/PhD_Thesis_Datasets/12_combined_cycle_power_plant/proces

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>